# Blinkit Data Analytics — Regional market
DuckDB SQL + Python | Yash Prajapati

## 1. Setup

### 1.1 Libraries

In [1]:
import warnings, math, textwrap
warnings.filterwarnings('ignore')
import duckdb, pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', lambda v: f'{v:,.2f}')
plt.rcParams.update({'figure.figsize':(10,4.5),'axes.grid':True,'grid.alpha':.25,'axes.spines.top':False,'axes.spines.right':False,'font.size':10})
print('duckdb', duckdb.__version__, '| pandas', pd.__version__, '| numpy', np.__version__)

duckdb 1.5.5 | pandas 3.0.2 | numpy 2.4.4


### 1.2 Load cleaned workbook

In [2]:
XLSX = 'data/Blinkit_analysis_new.xlsx'
book = pd.read_excel(XLSX, sheet_name=None)
geo = pd.read_csv('data/city_state_zone.csv')
for name, df in book.items():
    print(f'{name:28s} {df.shape[0]:>6,} rows  {df.shape[1]:>3} cols')

Data_Quality_Report              46 rows    3 cols
Orders_Customer_Info          5,000 rows   16 cols
Orders_Raw_Archive            5,000 rows   23 cols
Order_Line_Items              5,000 rows   14 cols
Delivery_Performance          5,000 rows    8 cols
Customer_Feedback             5,000 rows    8 cols
Customers                     2,500 rows   11 cols
Products                        268 rows   10 cols
Inventory_Analysis              268 rows   18 cols
Data_Dictionary                  44 rows    4 cols
Reference_Parameters             11 rows    5 cols
Relational_Diagram                0 rows    0 cols
Pivot_Payment_Method              7 rows    7 cols
Pivot_Customer_Segment            7 rows    5 cols
Pivot_Category_Sales             14 rows    4 cols
Pivot_Monthly_Trend              24 rows    6 cols
Pivot_Area_Delivery              23 rows    6 cols
Pivot_Inventory_Movement          6 rows    7 cols
Pivot_Category_Stock             14 rows    8 cols


### 1.3 Create DuckDB database and load tables

In [3]:
con = duckdb.connect('blinkit.duckdb')
load = {'orders_src':'Orders_Raw_Archive','items_src':'Order_Line_Items','delivery_src':'Delivery_Performance',
        'feedback_src':'Customer_Feedback','customers_src':'Customers','products_src':'Products','inventory_src':'Inventory_Analysis'}
for tbl, sheet in load.items():
    df = book[sheet].copy()
    df.columns = [c.strip() for c in df.columns]
    con.register('tmp_df', df)
    con.execute(f'CREATE OR REPLACE TABLE {tbl} AS SELECT * FROM tmp_df')
con.register('geo_df', geo)
con.execute('CREATE OR REPLACE TABLE geo AS SELECT * FROM geo_df')
con.execute("SELECT table_name, estimated_size FROM duckdb_tables() ORDER BY table_name").df()

,table_name,estimated_size
0,customers_src,2500
1,delivery_src,5000
2,feedback_src,5000
3,geo,316
4,inventory_src,268
5,items_src,5000
6,orders_src,5000
7,products_src,268


### 1.4 Typed analysis views

In [4]:
con.execute('''
CREATE OR REPLACE VIEW orders AS
SELECT order_id, customer_id,
       strptime(order_date, '%d-%m-%Y %H:%M')              AS order_ts,
       CAST(strptime(order_date, '%d-%m-%Y %H:%M') AS DATE) AS order_date,
       strptime(promised_delivery_time, '%d-%m-%Y %H:%M')   AS promised_ts,
       strptime(actual_delivery_time,  '%d-%m-%Y %H:%M')    AS actual_ts,
       delivery_status, order_total, payment_method, delivery_partner_id, store_id,
       CAST(delivery_time_minutes AS INTEGER)               AS delay_min,
       distance_km, reasons_if_delayed, customer_name,
       trim(area) AS area, pincode, customer_segment,
       CAST(registration_date AS DATE)                      AS registration_date,
       order_day_of_week, order_time_slot, order_value_segment,
       CASE WHEN is_weekend = 'Yes' THEN 1 ELSE 0 END       AS is_weekend
FROM orders_src ''')

con.execute('''
CREATE OR REPLACE VIEW items AS
SELECT order_id, product_id, quantity, unit_price, product_name, category, brand,
       price, mrp, margin_percentage, shelf_life_days, min_stock_level, max_stock_level, line_total,
       line_total * margin_percentage / 100.0 AS margin_value
FROM items_src ''')

con.execute('''
CREATE OR REPLACE VIEW feedback AS
SELECT feedback_id, order_id, customer_id, rating, feedback_category, sentiment,
       CAST(feedback_date AS DATE) AS feedback_date
FROM feedback_src ''')

con.execute('''
CREATE OR REPLACE VIEW f_sales AS
SELECT o.order_id, o.customer_id, o.order_ts, o.order_date,
       date_trunc('month', o.order_date)  AS order_month,
       extract(hour FROM o.order_ts)      AS order_hour,
       o.order_day_of_week, o.is_weekend, o.order_time_slot, o.order_value_segment,
       o.payment_method, o.customer_segment, o.registration_date, o.customer_name,
       o.area, g.state, g.zone, g.city_tier,
       i.product_id, i.product_name, i.category, i.brand,
       i.quantity, i.line_total AS revenue, i.margin_value, i.margin_percentage,
       i.price, i.mrp, i.shelf_life_days,
       o.delay_min, o.distance_km, o.delivery_status,
       CASE WHEN o.delivery_status = 'On Time' THEN 1 ELSE 0 END AS is_on_time_status,
       CASE WHEN o.delay_min > 0 THEN 1 ELSE 0 END               AS is_late_minutes,
       f.rating, f.sentiment, f.feedback_category
FROM orders o
JOIN items i    ON i.order_id = o.order_id
LEFT JOIN geo g ON g.area     = o.area
LEFT JOIN feedback f ON f.order_id = o.order_id ''')

con.execute('SELECT COUNT(*) AS fact_rows, COUNT(DISTINCT order_id) AS orders, MIN(order_date) AS first_day, MAX(order_date) AS last_day FROM f_sales').df()

,fact_rows,orders,first_day,last_day
0,5000,5000,2023-03-16,2024-11-04


### 1.5 Query helper

In [5]:
def q(sql, con=con):
    return con.execute(textwrap.dedent(sql)).df()

def pct(x, n):
    return round(100.0 * x / n, 2) if n else 0.0

q('SELECT COUNT(*) AS rows_in_fact_view FROM f_sales')

,rows_in_fact_view
0,5000


## 13. Region and zone

### 13.1 Zone summary

In [6]:
q('''
SELECT zone, COUNT(DISTINCT state) AS states, COUNT(DISTINCT area) AS cities,
       COUNT(*) AS orders, COUNT(DISTINCT customer_id) AS customers,
       round(SUM(revenue), 0) AS revenue,
       round(100 * SUM(revenue) / SUM(SUM(revenue)) OVER (), 2) AS revenue_share_pct,
       round(AVG(revenue), 0) AS aov,
       round(100.0 * COUNT(*) FILTER (WHERE is_on_time_status = 1) / COUNT(*), 2) AS on_time_pct,
       round(AVG(rating), 2) AS avg_rating
FROM f_sales GROUP BY zone ORDER BY revenue DESC ''')

,zone,states,cities,orders,customers,revenue,revenue_share_pct,aov,on_time_pct,avg_rating
0,South,6,87,1405,583,"1,392,807.00",28.01,991.00,69.82,3.35
1,East,4,70,1042,456,"1,043,890.00",20.99,"1,002.00",68.23,3.34
2,Central,3,59,1009,439,"992,872.00",19.97,984.00,68.68,3.41
3,West,2,46,695,317,"686,019.00",13.80,987.00,71.08,3.33
4,North,8,42,617,278,"622,453.00",12.52,"1,009.00",69.37,3.28
5,North East,5,12,232,99,"234,374.00",4.71,"1,010.00",70.26,3.30


### 13.2 State level metrics

In [7]:
states = q('''
WITH cust AS (
  SELECT state, customer_id, COUNT(DISTINCT order_id) AS orders FROM f_sales GROUP BY 1, 2)
SELECT s.state, any_value(s.zone) AS zone,
       COUNT(*) AS orders, COUNT(DISTINCT s.customer_id) AS customers,
       round(SUM(s.revenue), 0) AS revenue,
       round(100 * SUM(s.revenue) / SUM(SUM(s.revenue)) OVER (), 2) AS revenue_share_pct,
       round(AVG(s.revenue), 0) AS aov,
       round(100.0 * COUNT(*) FILTER (WHERE s.is_on_time_status = 1) / COUNT(*), 2) AS on_time_pct,
       round(AVG(s.rating), 2) AS avg_rating,
       round(100.0 * (SELECT COUNT(*) FROM cust c WHERE c.state = s.state AND c.orders >= 2)
             / NULLIF((SELECT COUNT(*) FROM cust c WHERE c.state = s.state), 0), 2) AS repeat_rate_pct
FROM f_sales s GROUP BY s.state ORDER BY revenue DESC ''')
states.head(15)

,state,zone,orders,customers,revenue,revenue_share_pct,aov,on_time_pct,avg_rating,repeat_rate_pct
0,Uttar Pradesh,Central,628,270,"624,328.00",12.56,994.00,68.31,3.44,70.37
1,Andhra Pradesh,South,556,228,"541,933.00",10.90,975.00,69.60,3.40,73.68
2,Maharashtra,West,491,219,"489,286.00",9.84,997.00,70.67,3.33,63.01
3,West Bengal,East,426,181,"422,573.00",8.50,992.00,69.72,3.39,70.72
4,Bihar,East,378,170,"365,706.00",7.35,967.00,67.99,3.26,70.00
5,Tamil Nadu,South,352,149,"333,319.00",6.70,947.00,67.90,3.28,69.80
6,Madhya Pradesh,Central,318,136,"305,666.00",6.15,961.00,71.70,3.33,72.06
7,Karnataka,South,217,91,"211,005.00",4.24,972.00,71.43,3.40,71.43
8,Gujarat,West,204,98,"196,733.00",3.96,964.00,72.06,3.32,61.22
9,Haryana,North,162,68,"176,774.00",3.56,"1,091.00",66.67,3.35,72.06


### 13.3 Composite priority score

In [8]:
scored = states[states.orders >= 50].copy()
for col in ['revenue_share_pct','repeat_rate_pct','on_time_pct']:
    scored[col + '_rank'] = scored[col].rank(pct=True)
scored['priority_score'] = (100 * (0.4*scored.revenue_share_pct_rank + 0.3*scored.repeat_rate_pct_rank
                                   + 0.3*scored.on_time_pct_rank)).round(0)
scored.sort_values('priority_score', ascending=False)[
    ['state','zone','orders','revenue','revenue_share_pct','repeat_rate_pct','on_time_pct','priority_score']].head(12)

,state,zone,orders,revenue,revenue_share_pct,repeat_rate_pct,on_time_pct,priority_score
1,Andhra Pradesh,South,556,"541,933.00",10.90,73.68,69.60,81.00
6,Madhya Pradesh,Central,318,"305,666.00",6.15,72.06,71.70,74.00
0,Uttar Pradesh,Central,628,"624,328.00",12.56,70.37,68.31,72.00
3,West Bengal,East,426,"422,573.00",8.50,70.72,69.72,70.00
7,Karnataka,South,217,"211,005.00",4.24,71.43,71.43,68.00
15,Kerala,South,106,"111,464.00",2.24,74.42,76.42,68.00
2,Maharashtra,West,491,"489,286.00",9.84,63.01,70.67,61.00
10,Rajasthan,North,178,"173,523.00",3.49,67.09,73.60,60.00
4,Bihar,East,378,"365,706.00",7.35,70.00,67.99,58.00
11,Telangana,South,148,"172,841.00",3.48,72.13,68.24,55.00


### 13.4 High revenue with weak delivery

In [9]:
med_rev = states.revenue.median(); med_ot = states.on_time_pct.median()
quad = states.assign(quadrant=np.where((states.revenue >= med_rev) & (states.on_time_pct < med_ot), 'high revenue, weak delivery',
                              np.where((states.revenue >= med_rev), 'high revenue, good delivery',
                              np.where((states.on_time_pct < med_ot), 'low revenue, weak delivery', 'low revenue, good delivery'))))
print(quad.groupby('quadrant').agg(states=('state','count'), revenue=('revenue','sum')).reset_index())
quad[quad.quadrant == 'high revenue, weak delivery'][['state','zone','orders','revenue','on_time_pct','repeat_rate_pct']]

                      quadrant  states      revenue
0  high revenue, good delivery       6 1,535,971.00
1  high revenue, weak delivery       8 2,805,358.00
2   low revenue, good delivery       8   395,795.00
3   low revenue, weak delivery       6   235,291.00


,state,zone,orders,revenue,on_time_pct,repeat_rate_pct
0,Uttar Pradesh,Central,628,"624,328.00",68.31,70.37
1,Andhra Pradesh,South,556,"541,933.00",69.60,73.68
3,West Bengal,East,426,"422,573.00",69.72,70.72
4,Bihar,East,378,"365,706.00",67.99,70.00
5,Tamil Nadu,South,352,"333,319.00",67.90,69.80
9,Haryana,North,162,"176,774.00",66.67,72.06
11,Telangana,South,148,"172,841.00",68.24,72.13
12,Jharkhand,East,155,"167,884.00",61.94,67.65


### 13.5 City tier performance

In [10]:
q('''
SELECT city_tier, COUNT(DISTINCT area) AS cities, COUNT(*) AS orders,
       round(SUM(revenue), 0) AS revenue, round(AVG(revenue), 0) AS aov,
       round(100.0 * COUNT(*) FILTER (WHERE is_on_time_status = 1) / COUNT(*), 2) AS on_time_pct,
       round(AVG(rating), 2) AS avg_rating
FROM f_sales GROUP BY city_tier ORDER BY revenue DESC ''')

,city_tier,cities,orders,revenue,aov,on_time_pct,avg_rating
0,Tier 3 - Small City,211,3410,"3,399,923.00",997.00,68.91,3.36
1,Tier 2 - Large City,96,1472,"1,473,073.00","1,001.00",70.58,3.31
2,Tier 1 - Metro,9,118,"99,420.00",843.00,68.64,3.38


### 13.6 Top cities

In [11]:
q('''
SELECT area AS city, any_value(state) AS state, COUNT(*) AS orders,
       round(SUM(revenue), 0) AS revenue, round(AVG(revenue), 0) AS aov,
       round(100.0 * COUNT(*) FILTER (WHERE is_on_time_status = 1) / COUNT(*), 2) AS on_time_pct
FROM f_sales GROUP BY area HAVING COUNT(*) >= 20 ORDER BY revenue DESC LIMIT 15 ''')

,city,state,orders,revenue,aov,on_time_pct
0,Orai,Uttar Pradesh,44,"44,503.00","1,011.00",61.36
1,Nandyal,Andhra Pradesh,36,"41,290.00","1,147.00",80.56
2,Gandhinagar,Gujarat,37,"37,951.00","1,026.00",72.97
3,Deoghar,Jharkhand,40,"37,749.00",944.00,70.00
4,Ghaziabad,Uttar Pradesh,32,"35,394.00","1,106.00",56.25
5,Nizamabad,Telangana,26,"33,225.00","1,278.00",65.38
6,Agra,Uttar Pradesh,27,"32,295.00","1,196.00",55.56
7,Sambalpur,Odisha,27,"32,138.00","1,190.00",66.67
8,Burhanpur,Madhya Pradesh,28,"31,549.00","1,127.00",64.29
9,Bathinda,Punjab,34,"31,476.00",926.00,70.59
